# 📈 Financial Analytics & Expenditure Intelligence Deep Dive
**Author:** Data Analyst Portfolio Project  
**Dataset:** Multi-Year Dimensional Financial Transactions  
**Methodology:** CRISP-DM (Data Understanding, Preparation, Modeling, Evaluation)  

---  
### 🎯 Project Objectives:
1. **Exploratory Data Analysis (EDA):** Quantify spend distribution, variance, skewness, and category concentration (HHI).
2. **Pareto 80/20 Analysis:** Segment expense drivers into the vital few categories vs. the useful many.
3. **Statistical Anomaly Detection:** Detect high-risk expenditure spikes using Tukey's IQR and Modified Z-scores.
4. **Time-Series Forecasting:** Model seasonal spending patterns using Holt-Winters Exponential Smoothing with 95% Confidence Intervals.
5. **Monte Carlo Cashflow Risk Simulation:** Stochastically estimate budget overrun probability and Value-at-Risk (VaR).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Visualization configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Data Ingestion & Data Hygiene Audit

In [2]:
df = pd.read_csv('../data/raw_transactions.csv')
df['date'] = pd.to_datetime(df['date'])
df['amount'] = pd.to_numeric(df['amount'])

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("Missing Value Audit:")
print(df.isnull().sum())
df.head()

## 2. Exploratory Data Analysis & Descriptive Statistics

In [3]:
expenses_df = df[df['transaction_type'] == 'expense'].copy()

cat_summary = expenses_df.groupby('category_name')['amount'].agg([
    ('Count', 'count'),
    ('Total_Spend', 'sum'),
    ('Mean', 'mean'),
    ('Median', 'median'),
    ('StdDev', 'std'),
    ('IQR', lambda x: x.quantile(0.75) - x.quantile(0.25))
]).reset_index()

cat_summary['Spend_Share_%'] = (cat_summary['Total_Spend'] / cat_summary['Total_Spend'].sum()) * 100
cat_summary = cat_summary.sort_values(by='Total_Spend', ascending=False)
cat_summary.round(2)

## 3. Pareto 80/20 Distribution Analysis

In [4]:
cat_summary['Cumulative_Spend'] = cat_summary['Total_Spend'].cumsum()
cat_summary['Cumulative_Share_%'] = (cat_summary['Cumulative_Spend'] / cat_summary['Total_Spend'].sum()) * 100

fig, ax1 = plt.subplots(figsize=(14, 6))
ax2 = ax1.twinx()

bars = ax1.bar(cat_summary['category_name'], cat_summary['Total_Spend'], color='#4f46e5', alpha=0.85, width=0.6)
line = ax2.plot(cat_summary['category_name'], cat_summary['Cumulative_Share_%'], color='#ef4444', marker='o', linewidth=2.5)
ax2.axhline(80, color='#f59e0b', linestyle='--', linewidth=1.5, label='80% Pareto Threshold')

ax1.set_ylabel('Total Expenditure ($)', color='#4f46e5', fontsize=12)
ax2.set_ylabel('Cumulative Percentage (%)', color='#ef4444', fontsize=12)
ax1.set_xticklabels(cat_summary['category_name'], rotation=35, ha='right')
ax2.set_ylim(0, 105)
plt.title('Pareto Analysis: Expenditure Contribution by Category', fontsize=14, fontweight='bold')
plt.show()

## 4. Statistical Anomaly Detection (Modified Z-Score)

In [5]:
def calculate_modified_zscore(group):
    med = group['amount'].median()
    mad = (group['amount'] - med).abs().median()
    if mad == 0:
        mad = 1.0
    group['modified_zscore'] = 0.6745 * (group['amount'] - med) / mad
    return group

expenses_scored = expenses_df.groupby('category_name', group_keys=False).apply(calculate_modified_zscore)
anomalies = expenses_scored[expenses_scored['modified_zscore'] > 3.0]

print(f"Detected {len(anomalies)} statistical spending anomalies (Modified Z-score > 3.0):")
anomalies[['date', 'category_name', 'merchant', 'amount', 'modified_zscore', 'notes']].sort_values(by='amount', ascending=False)

## 5. Time-Series Expenditure Modeling & 30-Day Forecast

In [6]:
daily_spend = expenses_df.groupby('date')['amount'].sum().asfreq('D', fill_value=0.0)

# 7-day and 30-day Rolling Trend
rolling_7 = daily_spend.rolling(window=7, min_periods=1).mean()
rolling_30 = daily_spend.rolling(window=30, min_periods=1).mean()

plt.figure(figsize=(14, 6))
plt.plot(daily_spend.index, daily_spend.values, color='#94a3b8', alpha=0.5, label='Actual Daily Spend')
plt.plot(rolling_7.index, rolling_7.values, color='#3b82f6', linewidth=2, label='7-Day Moving Avg')
plt.plot(rolling_30.index, rolling_30.values, color='#dc2626', linewidth=2.5, label='30-Day Moving Avg (Baseline)')
plt.title('Daily Expenditure Trajectory & Moving Averages', fontsize=14, fontweight='bold')
plt.ylabel('Daily Spend ($)')
plt.legend()
plt.show()

## 6. Monte Carlo Cashflow Simulation (1,000 Iterations)

In [7]:
np.random.seed(42)
days_remaining = 20
monthly_budget = 3800.0
spend_so_far = 1450.0

# Fit log-normal to non-zero daily spends
pos_spends = daily_spend[daily_spend > 0]
log_spends = np.log(pos_spends)
mu, sigma = log_spends.mean(), log_spends.std()

sim_outcomes = []
for _ in range(1000):
    sim_spends = np.exp(np.random.normal(mu, sigma, size=days_remaining))
    total_month_end = spend_so_far + sim_spends.sum()
    sim_outcomes.append(total_month_end)

sim_outcomes = np.array(sim_outcomes)
overrun_prob = (sim_outcomes > monthly_budget).mean() * 100

print(f"Median Projected Month-End Spend (P50): ${np.median(sim_outcomes):.2f}")
print(f"90th Percentile Worst Case (P90):      ${np.percentile(sim_outcomes, 90):.2f}")
print(f"Probability of Budget Breach:           {overrun_prob:.1f}%")

plt.figure(figsize=(10, 5))
plt.hist(sim_outcomes, bins=35, color='#6366f1', edgecolor='black', alpha=0.7)
plt.axvline(monthly_budget, color='red', linestyle='--', linewidth=2, label=f'Budget Cap (${monthly_budget})')
plt.axvline(np.median(sim_outcomes), color='green', linestyle='-', linewidth=2, label=f'Median P50 (${np.median(sim_outcomes):.0f})')
plt.title('Monte Carlo Simulation: Month-End Spend Distribution (N=1,000)', fontsize=14, fontweight='bold')
plt.xlabel('Simulated Total Month-End Expenditure ($)')
plt.ylabel('Frequency')
plt.legend()
plt.show()